# Kiểm tra dữ liệu và thiết kế tiền xử lý Student Dropout

Phạm vi: mô tả dữ liệu cho mục 3.1 và thiết kế tiền xử lý cho mục 3.3. Notebook kiểm tra dữ liệu, xuất bảng mô tả và khởi tạo bộ tiền xử lý; chưa chia Train/Test, fit hoặc đánh giá mô hình.

Nguồn: [UCI, dataset 697](https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success), DOI: 10.24432/C5MC89. Tài liệu mô tả: [Realinho et al. (2022)](https://www.mdpi.com/2306-5729/7/11/146).

Dữ liệu nguồn và thứ tự dòng được bảo toàn. Dữ liệu HK1/HK2 phản ánh thông tin sau nhập học; sử dụng toàn bộ đặc trưng không tương đương dự đoán tại thời điểm nhập học.

## 1. Môi trường và đường dẫn

Notebook sử dụng cấu hình đã lưu tại `configs/hieu_preprocessing_v1.json`. Các tệp nguồn và cấu hình chỉ được đọc; bảng kết quả được xuất vào `results/data_audit`. Thư mục dự án được xác định từ vị trí thực thi bên trong repo hoặc FinalExam.

In [1]:
import hashlib
import json
import sys
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

PROJECT_DIR = None
working_dir = Path.cwd().resolve()
for parent in (working_dir, *working_dir.parents):
    for candidate in (parent, parent / "FinalExam"):
        if (candidate / "src/hieu_data_preprocessing.py").is_file():
            PROJECT_DIR = candidate
            break
    if PROJECT_DIR is not None:
        break
if PROJECT_DIR is None:
    raise FileNotFoundError("Can chay notebook tu trong repo hoac FinalExam.")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

config_path = PROJECT_DIR / "configs/hieu_preprocessing_v1.json"
manifest_path = PROJECT_DIR / "data/source_manifest.json"
config = json.loads(config_path.read_text(encoding="utf-8"))
source_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
data_path = PROJECT_DIR / source_manifest["csv_path"]

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("Cau hinh:", config["config_id"], "-", config["status"])

pandas: 2.3.3
numpy: 2.2.6
scikit-learn: 1.9.1
Cau hinh: hieu_preprocessing_v1 - initial_design_not_evaluated


## 2. Nguồn dữ liệu, cấu trúc và kiểm tra thiếu/trùng

Đối chiếu SHA-256 của CSV với manifest và cấu hình đã ghi nhận. Nếu ZIP tải gốc có trên máy, kiểm tra thêm SHA-256 của ZIP và nội dung CSV bên trong. ZIP được giữ cục bộ; việc chạy lại bằng CSV không bắt buộc có ZIP.

Nạp CSV với dấu phân cách `;`. Chuẩn hóa khoảng trắng/tab ở đầu, cuối tên cột trên bản làm việc và tạo X/y một lần. Mỗi dòng là một sinh viên; Target gồm Dropout, Enrolled và Graduate. Target được loại khỏi X để tránh rò rỉ nhãn.

`isna()` kiểm tra giá trị thiếu được pandas nhận diện; `duplicated()` kiểm tra bản ghi trùng trên toàn bộ cột. Không tự động điền thiếu hoặc xóa bản ghi.

In [2]:
payload = data_path.read_bytes()
csv_sha256 = hashlib.sha256(payload).hexdigest()
assert csv_sha256 == source_manifest["csv_sha256"] == config["data_csv_sha256"], "CSV khong khop phien ban da ghi nhan."
zip_path = PROJECT_DIR / source_manifest["zip_path"]
if zip_path.exists():
    assert hashlib.sha256(zip_path.read_bytes()).hexdigest() == source_manifest["zip_sha256"]
    with zipfile.ZipFile(zip_path) as archive:
        assert payload == archive.read(source_manifest["csv_member_in_zip"])
    print("CSV khop ZIP nguon.")
else:
    print("Khong co ZIP cuc bo; CSV da duoc doi chieu SHA-256 voi manifest va cau hinh.")
print("CSV SHA-256:", csv_sha256)
print("Kich thuoc CSV (byte):", len(payload))

df = pd.read_csv(data_path, sep=";", encoding="utf-8-sig")
df_work = df.copy()
clean_names = df.columns.str.strip()
assert clean_names.is_unique, "Ten cot trung sau chuan hoa."
df_work.columns = clean_names
for old, new in zip(df.columns, df_work.columns, strict=True):
    if old != new:
        print("Chuan hoa ten cot:", repr(old), "->", repr(new))

X = df_work.drop(columns="Target")
y = df_work["Target"]
print("Kich thuoc du lieu:", df_work.shape)
print("Kich thuoc X/y:", X.shape, y.shape)
print("Tong so o thieu:", int(df.isna().sum().sum()))
print("So dong trung hoan toan:", int(df.duplicated().sum()))
print("Phan phoi nhan:")
print(y.value_counts())
display(df_work[["Course", "Age at enrollment", "Admission grade", "Target"]].head(3))

CSV khop ZIP nguon.
CSV SHA-256: 3ef126de5cefff26eb11fbb4237f1a1401cb64b488e2f1d598c23cedeb4c45ae
Kich thuoc CSV (byte): 533230
Chuan hoa ten cot: 'Daytime/evening attendance\t' -> 'Daytime/evening attendance'
Kich thuoc du lieu: (4424, 37)
Kich thuoc X/y: (4424, 36) (4424,)
Tong so o thieu: 0
So dong trung hoan toan: 0
Phan phoi nhan:
Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64


,Course,Age at enrollment,Admission grade,Target
0,171,20,127.3,Dropout
1,9254,19,142.5,Graduate
2,9070,19,124.8,Dropout


## 3. Bảng thuộc tính và phân nhóm theo ý nghĩa

Phân loại biến theo mô tả nguồn, không suy ra chỉ từ kiểu lưu trữ: 9 biến mã phân loại, 8 biến nhị phân, 1 biến thứ bậc và 18 biến định lượng. Application order là thứ tự nguyện vọng; Application mode là mã hình thức tuyển sinh.

Danh sách nhóm được đọc từ cấu hình tiền xử lý, tránh duy trì hai bộ danh sách độc lập. Application order giữ loại ngữ nghĩa thứ bậc, dù cấu hình ban đầu biểu diễn bằng One-Hot. Bảng thuộc tính ghi riêng tên gốc, tên chuẩn hóa, ý nghĩa và phương án xử lý dự kiến.

In [3]:
numeric_cols = config["numeric"]["columns"]
onehot_cols = config["categorical"]["columns"]
binary_cols = config["binary"]["columns"]
all_columns = numeric_cols + onehot_cols + binary_cols
assert len(all_columns) == len(set(all_columns)) == X.shape[1]
assert set(all_columns) == set(X.columns)

type_by_column = {
    **{column: "Dinh luong" for column in numeric_cols},
    **{column: "Ma phan loai" for column in onehot_cols},
    **{column: "Nhi phan" for column in binary_cols},
    "Application order": "Thu bac",
}
processing_by_column = {
    **{column: "StandardScaler" for column in numeric_cols},
    **{column: "OneHotEncoder" for column in onehot_cols},
    **{column: "Giu nguyen 0/1" for column in binary_cols},
}
feature_meanings = {
    "Marital status": "Tình trạng hôn nhân",
    "Application mode": "Hình thức đăng ký tuyển sinh",
    "Application order": "Thứ tự nguyện vọng",
    "Course": "Mã ngành/chương trình học",
    "Daytime/evening attendance": "Ca học ban ngày hoặc buổi tối",
    "Previous qualification": "Mã trình độ/bằng cấp trước khi nhập học",
    "Previous qualification (grade)": "Điểm của trình độ/bằng cấp trước khi nhập học",
    "Nacionality": "Quốc tịch",
    "Mother's qualification": "Trình độ học vấn của mẹ",
    "Father's qualification": "Trình độ học vấn của cha",
    "Mother's occupation": "Nghề nghiệp của mẹ",
    "Father's occupation": "Nghề nghiệp của cha",
    "Admission grade": "Điểm xét tuyển đầu vào",
    "Displaced": "Tình trạng di chuyển nơi ở để theo học",
    "Educational special needs": "Nhu cầu giáo dục đặc biệt",
    "Debtor": "Tình trạng nợ",
    "Tuition fees up to date": "Tình trạng cập nhật thanh toán học phí",
    "Gender": "Giới tính được mã hóa trong dữ liệu",
    "Scholarship holder": "Tình trạng nhận học bổng",
    "Age at enrollment": "Tuổi tại thời điểm nhập học",
    "International": "Tình trạng sinh viên quốc tế",
    "Unemployment rate": "Tỷ lệ thất nghiệp",
    "Inflation rate": "Tỷ lệ lạm phát",
    "GDP": "Chỉ số GDP trong dữ liệu nguồn",
}

semester_meanings = {
    "credited": "Số học phần được công nhận",
    "enrolled": "Số học phần đăng ký",
    "evaluations": "Số lượt đánh giá",
    "approved": "Số học phần đạt",
    "grade": "Điểm trung bình học kỳ",
    "without evaluations": "Số học phần không có đánh giá",
}
for semester, semester_number in [("1st", 1), ("2nd", 2)]:
    for measure, meaning in semester_meanings.items():
        column = f"Curricular units {semester} sem ({measure})"
        feature_meanings[column] = f"{meaning} ở học kỳ {semester_number}"

assert set(feature_meanings) == set(X.columns), "Mo ta thuoc tinh chua khop X."


raw_name_by_clean_name = dict(zip(df_work.columns, df.columns, strict=True))
schema = pd.DataFrame({
    "ten_cot_goc": [raw_name_by_clean_name[column] for column in X.columns],
    "ten_cot": X.columns,
    "y_nghia": [feature_meanings[column] for column in X.columns],
    "loai_bien": [type_by_column[column] for column in X.columns],
    "kieu_luu_tru": X.dtypes.astype(str).to_numpy(),
    "so_gia_tri_khac_nhau": X.nunique().to_numpy(),
    "so_o_thieu": X.isna().sum().to_numpy(),
    "tien_xu_ly_du_kien": [processing_by_column[column] for column in X.columns],
})
print("So cot theo nhom:")
print(schema["loai_bien"].value_counts())
display(schema.head(6))

So cot theo nhom:
loai_bien
Dinh luong      18
Ma phan loai     9
Nhi phan         8
Thu bac          1
Name: count, dtype: int64


,ten_cot_goc,ten_cot,y_nghia,loai_bien,kieu_luu_tru,so_gia_tri_khac_nhau,so_o_thieu,tien_xu_ly_du_kien
0,Marital status,Marital status,Tình trạng hôn nhân,Ma phan loai,int64,6,0,OneHotEncoder
1,Application mode,Application mode,Hình thức đăng ký tuyển sinh,Ma phan loai,int64,18,0,OneHotEncoder
2,Application order,Application order,Thứ tự nguyện vọng,Thu bac,int64,8,0,OneHotEncoder
3,Course,Course,Mã ngành/chương trình học,Ma phan loai,int64,17,0,OneHotEncoder
4,Daytime/evening attendance\t,Daytime/evening attendance,Ca học ban ngày hoặc buổi tối,Nhi phan,int64,2,0,Giu nguyen 0/1
5,Previous qualification,Previous qualification,Mã trình độ/bằng cấp trước khi nhập học,Ma phan loai,int64,17,0,OneHotEncoder


## 4. Miền giá trị nhị phân và mã Unknown

Kiểm tra tám cột nhị phân chỉ có giá trị 0/1. Theo từ điển UCI, mã 34 trong học vấn cha/mẹ biểu thị Unknown; mã này được thống kê riêng và dự kiến giữ như một nhóm phân loại, không tự chuyển thành NaN.

Kết quả đã xác minh: cả tám cột chỉ có 0/1; học vấn mẹ có 130 mã Unknown (2,94%), học vấn cha có 112 (2,53%). Hai thống kê tính riêng theo cột, không cộng để suy ra số sinh viên có thông tin chưa biết. Kiểm tra này chưa bao phủ miền mã của mọi cột phân loại.

In [4]:
binary_checks = []
for column in binary_cols:
    values = sorted(X[column].dropna().unique().tolist())
    invalid_count = int((X[column].notna() & ~X[column].isin([0, 1])).sum())
    missing_count = int(X[column].isna().sum())
    binary_checks.append({
        "ten_cot": column,
        "gia_tri_quan_sat": values,
        "so_gia_tri_ngoai_0_1": invalid_count,
        "so_o_thieu": missing_count,
    })

binary_check_table = pd.DataFrame(binary_checks)
print("KIEM TRA CAC COT NHI PHAN")
print(binary_check_table.to_string(index=False))

print("\nMA 34 (UNKNOWN) TRONG HOC VAN CHA/ME")
for column in ["Mother's qualification", "Father's qualification"]:
    count = int(X[column].eq(34).sum())
    print(f"{column}: {count} dong ({count / len(X):.2%})")

KIEM TRA CAC COT NHI PHAN
                   ten_cot gia_tri_quan_sat  so_gia_tri_ngoai_0_1  so_o_thieu
Daytime/evening attendance           [0, 1]                     0           0
                 Displaced           [0, 1]                     0           0
 Educational special needs           [0, 1]                     0           0
                    Debtor           [0, 1]                     0           0
   Tuition fees up to date           [0, 1]                     0           0
                    Gender           [0, 1]                     0           0
        Scholarship holder           [0, 1]                     0           0
             International           [0, 1]                     0           0

MA 34 (UNKNOWN) TRONG HOC VAN CHA/ME
Mother's qualification: 130 dong (2.94%)
Father's qualification: 112 dong (2.53%)


## 5. Tính hợp lệ của các thuộc tính định lượng

Kiểm tra thiếu/vô hạn trong 18 cột định lượng, tính nguyên và không âm của 10 cột đếm, tính nguyên và dương của tuổi nhập học. Giá trị min/max phục vụ đối chiếu miền dữ liệu; đây không phải EDA để lựa chọn mô hình.

Kết quả đã xác minh: không có NaN/vô hạn; các cột đếm đều nguyên và không âm; tuổi nhập học nguyên, trong khoảng 17–70. Inflation rate và GDP âm không tự động là lỗi. Điểm 0 không được tự coi là thiếu; giá trị cực trị chưa bị loại bỏ hoặc cắt ngưỡng.

In [5]:
numeric_quality_rows = []
for column in numeric_cols:
    series = X[column]
    numeric_quality_rows.append({
        "ten_cot": column,
        "min": series.min(),
        "max": series.max(),
        "so_o_thieu": int(series.isna().sum()),
        "so_vo_han": int(np.isinf(series.to_numpy()).sum()),
    })

numeric_quality = pd.DataFrame(numeric_quality_rows)
print("MIEN GIA TRI QUAN SAT")
print(numeric_quality.to_string(index=False))

count_cols = [
    column for column in numeric_cols
    if column.startswith("Curricular units") and not column.endswith("(grade)")
]
count_values = X[count_cols]

print("\nKIEM TRA CAC COT DEM")
print("So cot dem:", len(count_cols))
print("So o am:", int((count_values < 0).sum().sum()))
print("So o huu han khong nguyen:", int((
    np.isfinite(count_values) & count_values.mod(1).ne(0)
).sum().sum()))

age = X["Age at enrollment"]
print("\nKIEM TRA TUOI NHAP HOC")
print("So tuoi khong duong:", int(age.le(0).sum()))
print("So tuoi huu han khong nguyen:", int((
    np.isfinite(age) & age.mod(1).ne(0)
).sum()))

MIEN GIA TRI QUAN SAT
                                       ten_cot   min        max  so_o_thieu  so_vo_han
                Previous qualification (grade) 95.00 190.000000           0          0
                               Admission grade 95.00 190.000000           0          0
                             Age at enrollment 17.00  70.000000           0          0
                             Unemployment rate  7.60  16.200000           0          0
                                Inflation rate -0.80   3.700000           0          0
                                           GDP -4.06   3.510000           0          0
           Curricular units 1st sem (credited)  0.00  20.000000           0          0
           Curricular units 1st sem (enrolled)  0.00  26.000000           0          0
        Curricular units 1st sem (evaluations)  0.00  45.000000           0          0
           Curricular units 1st sem (approved)  0.00  26.000000           0          0
              Curricu

## 6. Bộ tiền xử lý dự kiến cho LR

Khởi tạo từ `configs/hieu_preprocessing_v1.json` bằng hàm dùng lại trong `src/hieu_data_preprocessing.py`:

- 18 cột định lượng: StandardScaler.
- 9 cột mã phân loại và Application order: OneHotEncoder, `handle_unknown="ignore"`.
- 8 cột nhị phân: giữ nguyên 0/1.

One-Hot cho Application order là lựa chọn ban đầu nhằm tránh áp đặt tác động tuyến tính theo khoảng cách giữa các bậc nguyện vọng. Mã Unknown có trong Train là một nhóm được học; giá trị chưa gặp lúc fit được encoder biểu diễn bằng toàn số 0 trong nhóm cột tương ứng.

Chưa fit bộ tiền xử lý. Khi đánh giá LR, toàn bộ preprocessing và bộ phân lớp phải được đặt trong Pipeline và fit riêng trên phần huấn luyện của từng fold. Không fit preprocessing trên toàn bộ dữ liệu trước khi chia tập. StandardScaler nhạy với ngoại lệ; cấu hình này chưa được đánh giá là tối ưu. Chưa áp dụng imputation, loại bỏ ngoại lệ hoặc resampling.

Tài liệu: [ColumnTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), [OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html), [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html), [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

In [6]:
from src.hieu_data_preprocessing import load_student_data, make_preprocessor

preprocessor = make_preprocessor(config_path)
for name, transformer, columns in preprocessor.transformers:
    method = transformer if isinstance(transformer, str) else type(transformer).__name__
    print(f"{name}: {len(columns)} cot -> {method}")
assert not hasattr(preprocessor, "transformers_")
print("Trang thai: da khoi tao, chua fit.")

numeric: 18 cot -> StandardScaler
categorical: 10 cot -> OneHotEncoder
binary: 8 cot -> passthrough
Trang thai: da khoi tao, chua fit.


## 7. Đối chiếu hàm đọc dữ liệu dùng lại

Đối chiếu kết quả từ `load_student_data` với quá trình đọc trực tiếp trong notebook, gồm giá trị, tên và thứ tự cột, chỉ mục, thứ tự dòng và kiểu dữ liệu. Tệp CSV gốc phải giữ nguyên SHA-256 sau quá trình kiểm tra.

In [7]:
X_from_module, y_from_module = load_student_data(data_path, config_path)
pd.testing.assert_frame_equal(X_from_module, X)
pd.testing.assert_series_equal(y_from_module, y)
assert hashlib.sha256(data_path.read_bytes()).hexdigest() == csv_sha256
print("X/y tu module khop notebook, bao gom thu tu dong va kieu du lieu.")
print("CSV goc khong thay doi.")

X/y tu module khop notebook, bao gom thu tu dong va kieu du lieu.
CSV goc khong thay doi.


## 8. Xuất bảng mô tả và phạm vi hoàn thành

Xuất một lần ba bảng vào `results/data_audit`: bảng 36 thuộc tính, phân phối ba nhãn và kiểm tra 18 thuộc tính định lượng. Cấu hình tiền xử lý và manifest nguồn không bị ghi đè khi chạy notebook.

Notebook này chỉ thực hiện kiểm tra dữ liệu và khởi tạo preprocessing phục vụ mục 3.1/3.3; không chia tập hoặc huấn luyện mô hình. Phát đã xác nhận cấu hình 80/20 stratify, seed 42 và CV 5-fold theo thông tin người dùng. Code và trạng thái kiểm chứng Dummy/LR được quản lý riêng; xem HIEU_README.md. Không dùng outputs của notebook này làm bằng chứng đã huấn luyện mô hình.

In [8]:
label_counts = y.value_counts()
target_distribution = pd.DataFrame({
    "nhan": label_counts.index,
    "so_mau": label_counts.to_numpy(),
    "ty_le": label_counts.to_numpy() / len(y),
})
output_dir = PROJECT_DIR / "results/data_audit"
output_dir.mkdir(parents=True, exist_ok=True)
for name, table in {
    "feature_dictionary.csv": schema,
    "target_distribution.csv": target_distribution,
    "numeric_quality.csv": numeric_quality,
}.items():
    table.to_csv(output_dir / name, index=False, encoding="utf-8-sig")
    print(f"Da luu results/data_audit/{name}: {len(table)} dong")
print("\nPHAN PHOI NHAN")
print(target_distribution.to_string(index=False))

Da luu results/data_audit/feature_dictionary.csv: 36 dong
Da luu results/data_audit/target_distribution.csv: 3 dong
Da luu results/data_audit/numeric_quality.csv: 18 dong

PHAN PHOI NHAN
    nhan  so_mau    ty_le
Graduate    2209 0.499322
 Dropout    1421 0.321203
Enrolled     794 0.179476
